Importing and cleaning list of largest companies by revenue

In [277]:
import sys
print(sys.version)

import pandas as pd
from bs4 import BeautifulSoup
import requests
from io import StringIO

3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]


In [278]:
url = 'https://en.wikipedia.org/wiki/List_of_largest_companies_by_revenue'

headers = {"User-Agent": "05_Assignment"}
resp = requests.get(url, headers=headers, timeout=10)
resp.raise_for_status()
html = resp.text
print(resp.status_code, len(html))
html[:500]

200 404155


'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp'

In [279]:
soup = BeautifulSoup(html, 'html.parser')

page_title = soup.title.text if soup.title else None
print(page_title)


List of largest companies by revenue - Wikipedia


In [280]:
h1_title = soup.find("h1", class_="firstHeading")
print("h1 =", h1_title.get_text(strip=True) if h1_title else None)

h1 = List of largest companies by revenue


In [281]:
links = [(l.text.strip(), l.get('href')) for l in soup.select("ul li a")]
links[:5]

[('Main page', '/wiki/Main_Page'),
 ('Contents', '/wiki/Wikipedia:Contents'),
 ('Current events', '/wiki/Portal:Current_events'),
 ('Random article', '/wiki/Special:Random'),
 ('About Wikipedia', '/wiki/Wikipedia:About')]

In [282]:
tables = pd.read_html(StringIO(html))
print(f"{len(tables)} tables found.")
tables[0].head(10)

6 tables found.


,Ranks,Name,Industry,Revenue,Profit,Employees,Headquarters[note 1],State-owned,Ref.
,Ranks,Name,Industry,USD (in billions),USD (in billions),Employees,Headquarters[note 1],State-owned,Ref.
0,1,Amazon,Retail Information technology,716,79.9,1576000,United States,NaN,[5]
1,2,Walmart,Retail,713,21.8,2100000,United States,NaN,[6]
2,3,State Grid Corporation of China,Electricity,545,9.2,1361423,China,NaN,[7]
3,4,Saudi Aramco,Oil and gas,480,106.0,73311,Saudi Arabia,NaN,[8]
4,5,China National Petroleum Corporation,Oil and gas,476,25.2,1026301,China,NaN,[9]
5,6,China Petrochemical Corporation,Oil and gas,429,9.3,513434,China,NaN,[10]
6,7,Apple,Information technology,416,112.0,166000,United States,NaN,[11]
7,8,Alphabet,Information technology,402,132.0,190820,United States,NaN,[12]
8,9,UnitedHealth Group,Healthcare,400,14.4,400000,United States,NaN,[13]


In [283]:
df = tables[0]
df.columns

MultiIndex([(               'Ranks',                'Ranks'),
            (                'Name',                 'Name'),
            (            'Industry',             'Industry'),
            (             'Revenue',    'USD (in billions)'),
            (              'Profit',    'USD (in billions)'),
            (           'Employees',            'Employees'),
            ('Headquarters[note 1]', 'Headquarters[note 1]'),
            (         'State-owned',          'State-owned'),
            (                'Ref.',                 'Ref.')],
           )

In [284]:
# Fix state_owned to be yes and no
state_owned = []

table = soup.find_all("table")[0]

for row in table.find_all("tr")[2:]:   # skip headers
    row_html = str(row)

    if "table-yes2" in row_html:
        state_owned.append(True)
    elif "table-no2" in row_html:
        state_owned.append(False)
print(len(df))
print(len(state_owned))
print(state_owned[:10])


50
50
[False, False, True, True, True, True, False, False, False, False]


In [285]:
df['State-owned'] = state_owned

In [286]:
df = tables[0].copy()

df.columns = [
    col[0] if col[0] == col[1] else f"{col[0]}_{col[1]}"
    for col in df.columns
]

df["State-owned"] = state_owned

df = df.rename(columns={'Headquarters[note 1]': 'Country'})
df = df.drop(columns=[
    "Ref."
])

df.head()

,Ranks,Name,Industry,Revenue_USD (in billions),Profit_USD (in billions),Employees,Country,State-owned
0,1,Amazon,Retail Information technology,716,79.9,1576000,United States,False
1,2,Walmart,Retail,713,21.8,2100000,United States,False
2,3,State Grid Corporation of China,Electricity,545,9.2,1361423,China,True
3,4,Saudi Aramco,Oil and gas,480,106.0,73311,Saudi Arabia,True
4,5,China National Petroleum Corporation,Oil and gas,476,25.2,1026301,China,True


In [287]:
df = df.rename(columns={'Revenue_USD (in billions)': 'Revenue (USD$)', 'Profit_USD (in billions)': 'Profit (USD$)'})
df['Revenue (USD$)'] = (df['Revenue (USD$)'].astype(int) * 1_000_000_000)
df["Profit (USD$)"] = (
    pd.to_numeric(df["Profit (USD$)"], errors="coerce")
    * 1_000_000_000
) # keeps the missing value in profit and leaves profit as float because it is the only missing value and it doesnt
# make sense to drop it or input the average or median profit for the table
df.head()

,Ranks,Name,Industry,Revenue (USD$),Profit (USD$),Employees,Country,State-owned
0,1,Amazon,Retail Information technology,716000000000,7.990000e+10,1576000,United States,False
1,2,Walmart,Retail,713000000000,2.180000e+10,2100000,United States,False
2,3,State Grid Corporation of China,Electricity,545000000000,9.200000e+09,1361423,China,True
3,4,Saudi Aramco,Oil and gas,480000000000,1.060000e+11,73311,Saudi Arabia,True
4,5,China National Petroleum Corporation,Oil and gas,476000000000,2.520000e+10,1026301,China,True


In [288]:
countries_needed = tables[1]['Country'].unique()
countries_needed

<StringArray>
[ 'United States',          'China',        'Germany', 'United Kingdom',
    'Switzerland',          'Japan',         'France',    'Netherlands',
    'South Korea',   'Saudi Arabia',      'Singapore',         'Taiwan']
Length: 12, dtype: str

Import and clean gdp data

In [289]:
gdp_url = 'https://www.worldometers.info/gdp/gdp-by-country/?source=wb&region=worldwide&year=2024&metric=nominal'

headers = {"User-Agent": "05_Assignment"}
resp = requests.get(gdp_url, headers=headers, timeout=10)
resp.raise_for_status()
html = resp.text
print(resp.status_code, len(html))
html[:500]

200 474894


'<!DOCTYPE html><html lang="en" dir="ltr" data-astro-transition-scope="astro-s7reefwf-5"> <head><meta charset="UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=edge"><meta name="viewport" content="width=device-width, initial-scale=1"><title>GDP by Country - Worldometer</title><!-- <I18nTags /> --><script async src="https://www.googletagmanager.com/gtag/js?id=G-ZDP3BFSX60"></script> <script>\n  window.dataLayer = window.dataLayer || [];\n  function gtag() {\n    // eslint-disable-next-line prefe'

In [290]:
soup = BeautifulSoup(html, 'html.parser')

page_title = soup.title.text if soup.title else None
print(page_title)

GDP by Country - Worldometer


In [291]:
gdp_tables = pd.read_html(StringIO(html))
print(len(gdp_tables))

2


In [292]:
gdp_tables[0].head()

,#,Country,GDP (US$),"GDP (Full Value, US$)",GDP Growth,GDP per Capita (US$)
0,1,United States,$29.3 trillion,"$29,298,025,000,000",2.79%,"$86,173"
1,2,China,$18.95 trillion,"$18,945,112,000,000",5.00%,"$13,453"
2,3,Germany,$4.68 trillion,"$4,684,182,000,000",â0.50%,"$56,087"
3,4,Japan,$4.19 trillion,"$4,190,008,000,000",â0.24%,"$33,820"
4,5,India,$3.76 trillion,"$3,760,814,000,000",7.10%,"$2,592"


In [293]:
gdp_df = gdp_tables[1].copy()

In [294]:
gdp_df = gdp_df[gdp_df['Country'].isin(countries_needed)]
gdp_df.head()

,#,Country,GDP (US$),"GDP (Full Value, US$)",GDP Growth,GDP per Capita (US$),Share of World GDP
0,1,United States,$29.3 trillion,"$29,298,013,000,000",2.79%,"$86,170",26.24%
1,2,China,$18.73 trillion,"$18,729,668,435,848",4.96%,"$13,293",16.77%
2,3,Germany,$4.69 trillion,"$4,685,592,577,805",â0.50%,"$56,104",4.20%
3,4,Japan,$4.19 trillion,"$4,190,008,188,359",â0.24%,"$33,797",3.75%
5,6,United Kingdom,$3.7 trillion,"$3,695,539,513,534",1.08%,"$53,341",3.31%


In [295]:
gdp_df["GDP Growth"] = (
    gdp_df["GDP Growth"]
    .astype(str)
    .str.replace("â", "-", regex=False) #strange character from the website
)
gdp_df.head()


,#,Country,GDP (US$),"GDP (Full Value, US$)",GDP Growth,GDP per Capita (US$),Share of World GDP
0,1,United States,$29.3 trillion,"$29,298,013,000,000",2.79%,"$86,170",26.24%
1,2,China,$18.73 trillion,"$18,729,668,435,848",4.96%,"$13,293",16.77%
2,3,Germany,$4.69 trillion,"$4,685,592,577,805",-0.50%,"$56,104",4.20%
3,4,Japan,$4.19 trillion,"$4,190,008,188,359",-0.24%,"$33,797",3.75%
5,6,United Kingdom,$3.7 trillion,"$3,695,539,513,534",1.08%,"$53,341",3.31%


In [296]:
gdp_df = gdp_df.drop(columns=[
    "#",
    "GDP (US$)"
])
gdp_df.head()

,Country,"GDP (Full Value, US$)",GDP Growth,GDP per Capita (US$),Share of World GDP
0,United States,"$29,298,013,000,000",2.79%,"$86,170",26.24%
1,China,"$18,729,668,435,848",4.96%,"$13,293",16.77%
2,Germany,"$4,685,592,577,805",-0.50%,"$56,104",4.20%
3,Japan,"$4,190,008,188,359",-0.24%,"$33,797",3.75%
5,United Kingdom,"$3,695,539,513,534",1.08%,"$53,341",3.31%


In [304]:
gdp_df["GDP (Full Value, US$)"] = (
    gdp_df["GDP (Full Value, US$)"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype("int64")
)
gdp_df["GDP Growth"] = (
    gdp_df["GDP Growth"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)
gdp_df["GDP per Capita (US$)"] = (
    gdp_df["GDP per Capita (US$)"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype("int64")
)
gdp_df["Share of World GDP"] = (
    gdp_df["Share of World GDP"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)
gdp_df = gdp_df.rename(columns={'GDP Growth': 'GDP Growth (%)', 'Share of World GDP': 'Share of World GDP (%)'})
gdp_df.head()

,Country,"GDP (Full Value, US$)",GDP Growth (%),GDP per Capita (US$),Share of World GDP (%)
0,United States,29298013000000,2.79,86170,26.24
1,China,18729668435848,4.96,13293,16.77
2,Germany,4685592577805,-0.50,56104,4.20
3,Japan,4190008188359,-0.24,33797,3.75
5,United Kingdom,3695539513534,1.08,53341,3.31


Merge

In [305]:
final_df = pd.merge(df, gdp_df, left_on="Country", right_on="Country", how="left")
print(final_df.dtypes)
final_df.head()

Ranks                       int64
Name                          str
Industry                      str
Revenue (USD$)              int64
Profit (USD$)             float64
Employees                   int64
Country                       str
State-owned                  bool
GDP (Full Value, US$)       int64
GDP Growth (%)            float64
GDP per Capita (US$)        int64
Share of World GDP (%)    float64
dtype: object


,Ranks,Name,Industry,Revenue (USD$),Profit (USD$),Employees,Country,State-owned,"GDP (Full Value, US$)",GDP Growth (%),GDP per Capita (US$),Share of World GDP (%)
0,1,Amazon,Retail Information technology,716000000000,7.990000e+10,1576000,United States,False,29298013000000,2.79,86170,26.24
1,2,Walmart,Retail,713000000000,2.180000e+10,2100000,United States,False,29298013000000,2.79,86170,26.24
2,3,State Grid Corporation of China,Electricity,545000000000,9.200000e+09,1361423,China,True,18729668435848,4.96,13293,16.77
3,4,Saudi Aramco,Oil and gas,480000000000,1.060000e+11,73311,Saudi Arabia,True,1254140800000,2.65,35528,1.12
4,5,China National Petroleum Corporation,Oil and gas,476000000000,2.520000e+10,1026301,China,True,18729668435848,4.96,13293,16.77


Save

In [306]:
df.to_csv("clean.csv", index=False)